In [1]:
from manim import *
import numpy as np

In [6]:
from __future__ import annotations
from manim import *
import numpy as np

# --- 1. Enhanced Spring Class ---
class TensionSpring(VMobject):
    """
    A procedural spring that connects two points.
    Designed to be used with always_redraw for dynamic animations.
    Use the .start and .end attributes instead of .start_point/.end_point to avoid attribute errors.
    """
    def __init__(self, start: np.ndarray, end: np.ndarray, 
                 coils: int = 8, radius: float = 0.1, color = RED, **kwargs):
        super().__init__(color=color, **kwargs)
        self.start = np.array(start)
        self.end = np.array(end)
        self.coils = coils
        self.radius = radius
        self.generate_points()

    def generate_points(self):
        vec = self.end - self.start
        length = np.linalg.norm(vec)
        
        # Handle edge case where start == end
        if length < 1e-6:
            self.set_points_as_corners([self.start, self.end])
            return

        direction = vec / length
        # Vector perpendicular to direction (for the coil width)
        perp = np.array([-direction[1], direction[0], 0])
        
        # Create zigzag pattern
        t_range = np.linspace(0, 1, self.coils * 12 + 1)
        points = []
        for t in t_range:
            sine_wave = self.radius * np.sin(2 * np.pi * self.coils * t) * perp
            linear_progress = self.start + (direction * length * t)
            points.append(linear_progress + sine_wave)
            
        self.set_points_as_corners(points)

# --- 2. Helper for Dynamic Redrawing ---
def get_dynamic_spring(mobj1: Mobject, mobj2: Mobject, color=RED) -> Mobject:
    """Returns a spring that automatically updates when mobj1 or mobj2 moves."""
    return always_redraw(lambda: TensionSpring(
        start=mobj1.get_top() + DOWN*0.05, # Anchor slightly inside
        end=mobj2.get_top() + DOWN*0.05,
        coils=10,
        radius=0.08,
        color=color,
        stroke_width=3
    ))

In [8]:
%%manim -v WARNING --fps 30 -ql RankingLossScene

class RankingLossScene(Scene):
    def construct(self):
        self.camera.background_color = "#1e1e1e"
        
        # --- Data Setup ---
        # 5 items. Index 2 (Blue) and Index 3 (Orange) are the conflict.
        # Blue (Rel=1) has score 2.5 (Too Low)
        # Orange (Rel=0) has score 3.5 (Too High)
        scores = np.array([2.0, 1.5, 2.5, 3.5, 1.0]) 
        relevance = np.array([1, 1, 1, 0, 0]) 
        
        # --- Visual Setup ---
        bar_width = 0.6
        spacing = 1.0
        
        bars = VGroup()
        for i, score in enumerate(scores):
            # Color logic: Relevant = Blue, Irrelevant = Orange
            color = BLUE if relevance[i] == 1 else ORANGE
            
            bar = Rectangle(
                width=bar_width, height=score, 
                fill_color=color, fill_opacity=0.8, stroke_color=WHITE, stroke_width=1
            )
            # Position logic
            x = (i - 2) * spacing
            bar.move_to([x, -2, 0], aligned_edge=DOWN)
            
            # Label
            lbl = Text(f"s_{i}", font_size=20, color=LIGHT_GREY).next_to(bar, DOWN)
            val = DecimalNumber(score, num_decimal_places=1, font_size=20).next_to(bar, UP)
            
            bars.add(VGroup(bar, lbl, val))
            
        # Ground line
        ground = Line(LEFT*4, RIGHT*4, color=GREY).move_to(DOWN*2)
        
        # Formula with color coding
        formula = MathTex(
            r"\mathcal{L}_{rank} = \sum \max(0, 1 - (",
            r"s_{pos}", 
            r"-", 
            r"s_{neg}",
            r"))"
        ).to_edge(UP)
        formula[1].set_color(BLUE)
        formula[3].set_color(ORANGE)
        
        self.add(ground)
        self.play(
            LaggedStart(*[GrowFromEdge(b[0], DOWN) for b in bars], lag_ratio=0.1),
            Write(formula)
        )
        self.play(FadeIn(VGroup(*[b[1:] for b in bars]))) # Fade in labels
        self.wait(0.5)

        # --- Identify Conflict ---
        idx_pos, idx_neg = 2, 3
        group_pos = bars[idx_pos] # The Blue one
        group_neg = bars[idx_neg] # The Orange one
        
        # Dim others
        others = VGroup(*[b for i, b in enumerate(bars) if i not in [idx_pos, idx_neg]])
        self.play(others.animate.set_opacity(0.3))
        
        # Highlight pair
        self.play(
            group_pos.animate.scale(1.1),
            group_neg.animate.scale(1.1),
            Indicate(group_pos[0], color=BLUE_A),
            Indicate(group_neg[0], color=ORANGE),
        )

        # --- Physics Animation ---
        
        # 1. Attach Dynamic Spring
        # Note: We attach to the bar rect (index 0 of the group)
        # Defensive: Make sure group_pos[0] and group_neg[0] have .get_top()
        # Remove any code referencing a missing '.end' attribute.
        spring = get_dynamic_spring(group_pos[0], group_neg[0])
        
        violation_text = Text("Violation!", font_size=24, color=RED).next_to(spring, UP, buff=0.5)
        
        self.play(Create(spring))
        self.play(Write(violation_text))
        self.play(spring[0].animate.set_stroke(width=5), rate_func=wiggle)
        
        # 2. Animate the Swap (Spring stretches/shrinks automatically)
        pos_x = group_pos.get_x()
        neg_x = group_neg.get_x()
        
        # Arc path to look like they are jumping over each other
        self.play(
            group_pos.animate.set_x(neg_x),
            group_neg.animate.set_x(pos_x),
            FadeOut(violation_text),
            run_time=1.5,
            rate_func=smooth
        )
        
        # 3. Snap Spring Removal
        self.play(FadeOut(spring), scale=0.5)
        
        # 4. Resolve & Restore
        check = Text("Fixed!", color=GREEN, font_size=24).next_to(group_pos, UP, buff=0.5)
        self.play(
            others.animate.set_opacity(1),
            group_pos.animate.scale(1/1.1),
            group_neg.animate.scale(1/1.1),
            Transform(violation_text, check)
        )
        self.wait(2)

Manim Community v0.19.1

AttributeError: TensionSpring object has no attribute 'end'

In [18]:
%%manim -v WARNING --fps 30 -ql UncertaintyLossScene

class UncertaintyLossScene(Scene):
    def construct(self):
        # 1. Setup Axes
        axes = Axes(
            x_range=[-2, 6, 1], y_range=[0, 1.5, 0.5],
            axis_config={"include_tip": False}
        ).scale(0.8).shift(DOWN*0.5)
        
        # 2. Trackers for animation
        mu_t = ValueTracker(0.0) # Bad prediction (far left)
        sigma_t = ValueTracker(1.2) # High uncertainty
        gt_x = 3.5
        
        # 3. Mobjects
        cloud_grp = create_gaussian_plot(axes, mu_t, sigma_t)
        
        gt_line = DashedLine(
            axes.c2p(gt_x, 0), axes.c2p(gt_x, 1.5), 
            color=YELLOW, stroke_width=4
        )
        gt_lbl = Text("Ground Truth", font_size=20, color=YELLOW).next_to(gt_line, UP)
        
        title = Title("Uncertainty Loss (NLL)").scale(0.8)
        
        self.add(title, axes, gt_line, gt_lbl)
        self.play(FadeIn(cloud_grp))
        self.wait(1)
        
        # 4. Phase 1: Shift Mean (Accuracy)
        arrow = Arrow(axes.c2p(0, 0.5), axes.c2p(3, 0.5), color=RED)
        txt = Text("Minimize Error", font_size=24, color=RED).next_to(arrow, UP)
        
        self.play(GrowArrow(arrow), Write(txt))
        self.play(
            mu_t.animate.set_value(gt_x),
            FadeOut(arrow), FadeOut(txt),
            run_time=2,
            rate_func=smooth
        )
        
        # 5. Phase 2: Shrink Variance (Precision)
        # Note: We clamp scale to avoid division by zero artifacts
        arrows = VGroup(
            Arrow(axes.c2p(gt_x - 1.5, 0.3), axes.c2p(gt_x - 0.5, 0.3), color=PURPLE),
            Arrow(axes.c2p(gt_x + 1.5, 0.3), axes.c2p(gt_x + 0.5, 0.3), color=PURPLE)
        )
        txt_2 = Text("Minimize Variance", font_size=24, color=PURPLE).next_to(cloud_grp, LEFT)
        
        self.play(FadeIn(arrows), Write(txt_2))
        self.play(
            sigma_t.animate.set_value(0.4),
            FadeOut(arrows), FadeOut(txt_2),
            run_time=2
        )
        
        final_txt = Text("High Confidence Match", font_size=32, color=GREEN).to_edge(DOWN)
        self.play(Write(final_txt))
        self.wait(2)

Manim Community v0.19.1

In [20]:
%%manim -v WARNING --fps 30 -ql CombinedLossScene

class CombinedLossScene(Scene):
    """
    Demonstrates that Ranking and Uncertainty run in parallel.
    Uses the helper functions to build the view without nesting Scenes.
    """
    def construct(self):
        # Layout Division
        self.play(Write(Text("Total Loss = Ranking + Uncertainty", font_size=36).to_edge(UP)))
        
        left_zone = VGroup().to_edge(LEFT, buff=1)
        right_zone = VGroup().to_edge(RIGHT, buff=1)
        
        # --- LEFT: Ranking Setup ---
        scores = [2, 4, 3] 
        relevance = [1, 0, 0] # Item 0 is relevant but score is low
        r_layout, r_bars = create_ranking_mobjects(scores, relevance)
        r_layout.scale(0.7).move_to(LEFT * 3.5 + DOWN)
        r_title = Text("Ranking", font_size=24, color=ORANGE).next_to(r_layout, UP)
        
        # --- RIGHT: Uncertainty Setup ---
        axes = Axes(x_range=[0, 5, 1], y_range=[0, 2, 1], x_length=4, y_length=3).scale(0.8)
        axes.move_to(RIGHT * 3.5 + DOWN)
        mu_t = ValueTracker(1.0)
        sigma_t = ValueTracker(0.8)
        gt_x = 3.0
        
        u_plot = create_gaussian_plot(axes, mu_t, sigma_t, color=BLUE)
        u_gt = Dot(axes.c2p(gt_x, 0), color=YELLOW, radius=0.1)
        u_title = Text("Uncertainty", font_size=24, color=BLUE).next_to(axes, UP)
        
        # Add Initial State
        self.play(
            FadeIn(r_layout), Write(r_title),
            FadeIn(axes), FadeIn(u_plot), FadeIn(u_gt), Write(u_title)
        )
        
        # --- ANIMATE BOTH SIMULTANEOUSLY ---
        
        # 1. Prepare Ranking Animation (Swap bars 0 and 1)
        bar_relevant = r_bars[0]
        bar_irrelevant = r_bars[1]
        
        # Create Spring
        spring = TensionSpring(bar_relevant[0].get_top(), bar_irrelevant[0].get_top(), color=RED)
        
        # 2. Execute Parallel Optimization
        self.play(Create(spring), run_time=1)
        
        self.play(
            # Ranking Action: Swap positions
            bar_relevant.animate.set_x(bar_irrelevant.get_x()),
            bar_irrelevant.animate.set_x(bar_relevant.get_x()),
            FadeOut(spring),
            
            # Uncertainty Action: Move Mu to GT and Shrink Sigma
            mu_t.animate.set_value(gt_x),
            sigma_t.animate.set_value(0.3),
            
            run_time=3,
            rate_func=smooth
        )
        
        self.wait(2)

Manim Community v0.19.1